### All data

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import truncnorm

np.random.seed(42)

this_dir = os.getcwd()
work_dir = os.path.dirname(this_dir)
data_dir = os.path.join(work_dir, 'data')
norm_data_dir = os.path.join(work_dir, 'data_norm')

os.makedirs(data_dir, exist_ok=True)
os.makedirs(norm_data_dir, exist_ok=True)

consonant_list = ['ts', 'tc', 's', 'c']
consonant_cog = [4000, 7000, 4000, 7000]
consonant_fd = [96, 96, 174, 174]


norm_refs = ['NA', 'NA', 'NA', 'NA', 'NA', 'NA', 'NA', 300, 200, 
             80, 200, 170, 110, 90, 3085.35, 1984.3, 626.2, 
             5500, 135, 200, 0.5, 1, 50, 60, 300, 200, 
             'NA', 'NA', 'NA', 'NA', 'NA', 'NA', 'NA', 'NA', 
             'NA', 'NA', 'NA', 'NA', 'NA', 'NA', 'NA', 300, 200, 
             80, 200, 170, 110, 90, 3085.35, 1984.3, 626.2]

# no. of tokens for each word
sample_size = 8000

metadata = []
norm_metadata = []

for index in range (4):
    consonant = consonant_list[index]
    cog = consonant_cog[index]
    fri_dur = consonant_fd[index]

    vowel = 'a'
    f_vals = [2815 ,1551 ,936]

    word = vowel + consonant + vowel

    if (index in [0, 3]) and (jndex in [0, 2, 4, 6]):
        istrain = 'yes'
    else:
        istrain = 'no'

    c_f_means = np.array([cog, fri_dur])
    c_f_stds = np.array([500, 13])

    con_means = np.array([200, 0.5, 1, 50, 60]) # sta_dev, skewness, kurtosis, bur_int, fri_int
    con_stds = con_means * 0.05

    con_means = np.concatenate((c_f_means, con_means))
    con_stds = np.concatenate((c_f_stds, con_stds))

    consonants = np.zeros((sample_size, len(con_means)))
    for i in range(len(con_means)):
        a, b = - 2*con_stds[i] / con_stds[i], 2*con_stds[i] / con_stds[i]
        dist = truncnorm(a, b, loc = con_means[i], scale = con_stds[i])
        consonants[:, i] = dist.rvs(size = sample_size)
    
    # total duration of fixed value 200 for consonants
    con_dur = np.full((sample_size, 1), 200)

    # zero values for all other features
    zeros = np.full((sample_size, 9), 0)

    # all concatenated
    consonants = np.hstack((consonants, con_dur, zeros))

    vow_means = np.array([80, 200, 170, 110 ,90]) # voc_int, f0, b3, b2, b1
    vow_means = np.append(vow_means, f_vals) # *, f3, f2, f1
    vow_stds = vow_means * 0.05

    vowels = np.zeros((sample_size, len(vow_means)))
    for i in range(len(vow_means)):
        a, b = - 2*vow_stds[i] / vow_stds[i], 2*vow_stds[i] / vow_stds[i]
        dist = truncnorm(a, b, loc = vow_means[i], scale = vow_stds[i])
        vowels[:, i] = dist.rvs(size = sample_size)

    # vocalic duration and total duration of fixed value 400 for vowels
    vow_dur = np.full((sample_size, 2), 400)

    # zero values for all other features
    zeros = np.full((sample_size, 7), 0)

    # all concatenated
    vowels = np.hstack((zeros, vow_dur, vowels))

    subdata_dir = os.path.join(data_dir, word)
    os.makedirs(subdata_dir, exist_ok=True)

    norm_subdata_dir = os.path.join(norm_data_dir, word)
    os.makedirs(norm_subdata_dir, exist_ok=True)

    for i in range(sample_size):
        uid = word + f'_{i+1:04d}'
        filename = f'{uid}.npy'
        save_path = os.path.join(subdata_dir, filename)
        
        # 1. raw data
        vcv = np.vstack([vowels[i], consonants[i], vowels[i]])
        np.save(save_path, vcv)

        # 2. norm data
        norm_save_path = os.path.join(norm_subdata_dir, filename)
        norm_vcv = vcv.copy()
        for row_idx in range(norm_vcv.shape[0]):
            for col_idx in range(norm_vcv.shape[1]):
                ref_value = norm_refs[row_idx * 17 + col_idx]
                if ref_value == 'NA':
                    norm_vcv[row_idx, col_idx] = 0
                else:
                    norm_vcv[row_idx, col_idx] = (norm_vcv[row_idx, col_idx] / ref_value) - 1
        np.save(norm_save_path, norm_vcv)

        # 3. metadata files
        save_path_rel = os.path.relpath(save_path, start=work_dir)
        metadata.append({
            'uid': uid,
            'path': save_path_rel,
            'cog': vcv[1][0],
            'fri_dur': vcv[1][1],
            'f1': vcv[0][-1],
            'f2': vcv[0][-2],
            'f3': vcv[0][-3],
            'word': word,
            'consonant': consonant,
            'vowel': vowel,
            'train': istrain
        })

        norm_save_path_rel = os.path.relpath(norm_save_path, start=work_dir)
        norm_metadata.append({
            'uid': uid,
            'path': norm_save_path_rel,
            'cog': norm_vcv[1][0],
            'fri_dur': norm_vcv[1][1],
            'f1': norm_vcv[0][-1],
            'f2': norm_vcv[0][-2],
            'f3': norm_vcv[0][-3],
            'word': word,
            'consonant': consonant,
            'vowel': vowel,
            'train': istrain
        })

csv_name = 'metadata.csv'
csv_path = os.path.join(data_dir, csv_name)
metaframe = pd.DataFrame(metadata)
metaframe.to_csv(csv_path, index=False)

norm_csv_name = 'norm_metadata.csv'
norm_csv_path = os.path.join(norm_data_dir, norm_csv_name)
norm_metaframe = pd.DataFrame(norm_metadata)
norm_metaframe.to_csv(norm_csv_path, index=False)


In [7]:
savetest = np.load(save_path)
print(savetest)

savetest = np.load(norm_save_path)
print(savetest)

[[0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 4.00000000e+02
  4.00000000e+02 8.29308996e+01 1.99760273e+02 1.79552239e+02
  1.14265773e+02 9.15326853e+01 2.98373450e+03 1.16668291e+03
  7.48867907e+02]
 [6.65778618e+03 1.70221202e+02 1.81112683e+02 4.60560255e-01
  1.00853328e+00 4.72338549e+01 6.08201071e+01 2.00000000e+02
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 4.00000000e+02
  4.00000000e+02 8.29308996e+01 1.99760273e+02 1.79552239e+02
  1.14265773e+02 9.15326853e+01 2.98373450e+03 1.16668291e+03
  7.48867907e+02]]
[[ 0.          0.          0.          0.          0.          0.
   0.          0.33333333  1.          0.03663625 -0.00119864  0.05618964
   0.03877975  0.01702984 -0.03293484 -0.41204308  0.19589254

vowel_list = ['i', 'L', 'e', 'F', 'u', 'W', 'o', 'D']
vowel_formants = [
    [3372, 2761, 437],
    [3053, 2365, 483],
    [3047, 2530, 536],
    [2979, 2058, 731],
    [2735, 1105, 459],
    [2827, 1225, 519],
    [2828, 1035, 555],
    [2824, 1136, 781]
]